## Karnataka

In [ ]:
import xarray as xr
import pandas as pd

# Open dataset
ds = xr.open_dataset("data.nc")

# Kelvin → Celsius
ds['t2m_c'] = ds['t2m'] - 273.15

# Daily maximum temperature
daily_max = ds['t2m_c'].resample(
    valid_time='1D'
).max()

# Convert raster → dataframe
df = daily_max.to_dataframe().reset_index()

# Remove unnecessary column
if 'number' in df.columns:
    df = df.drop(columns=['number'])

# Rename columns
df.rename(columns={
    'valid_time': 'date',
    't2m_c': 'max_temp'
}, inplace=True)

# Sort values
df = df.sort_values(
    ['latitude', 'longitude', 'date']
)

# Previous day temperatures
df['tmax_prev1'] = df.groupby(
    ['latitude', 'longitude']
)['max_temp'].shift(1)

df['tmax_prev2'] = df.groupby(
    ['latitude', 'longitude']
)['max_temp'].shift(2)

df['tmax_prev3'] = df.groupby(
    ['latitude', 'longitude']
)['max_temp'].shift(3)

# Rolling features
df['rolling_3day_avg'] = (
    df.groupby(['latitude', 'longitude'])['max_temp']
    .rolling(3)
    .mean()
    .reset_index(level=[0,1], drop=True)
)

df['rolling_3day_max'] = (
    df.groupby(['latitude', 'longitude'])['max_temp']
    .rolling(3)
    .max()
    .reset_index(level=[0,1], drop=True)
)

df['rolling_3day_std'] = (
    df.groupby(['latitude', 'longitude'])['max_temp']
    .rolling(3)
    .std()
    .reset_index(level=[0,1], drop=True)
)

# Temperature change
df['temp_change_3day'] = (
    df['max_temp'] - df['tmax_prev3']
)

# OPTIONAL: Normal temperature
df['normal_temp'] = df.groupby(
    ['latitude', 'longitude']
)['max_temp'].transform('mean')

# OPTIONAL: Departure / anomaly
df['departure'] = (
    df['max_temp'] - df['normal_temp']
)

# Heatwave label
df['heatwave'] = (
    (df['max_temp'] >= 40) &
    (df['departure'] >= 4)
).astype(int)

# Remove NaN rows
df = df.dropna()

# Save CSV
df.to_csv("karnataka_heatwave.csv", index=False)

print(df.head())

print("CSV saved successfully")

## Rajasthan

In [1]:
import xarray as xr
import pandas as pd
import numpy as np

# ============================================
# OPEN DATASET
# ============================================

ds = xr.open_dataset("rajasthan_data.nc")

# ============================================
# KELVIN → CELSIUS
# ============================================

ds['t2m_c'] = ds['t2m'] - 273.15

# ============================================
# DAILY MAX TEMPERATURE
# ============================================

daily_max = ds['t2m_c'].resample(
    valid_time='1D'
).max()

# ============================================
# RASTER → DATAFRAME
# ============================================

df = daily_max.to_dataframe().reset_index()

# ============================================
# REMOVE UNUSED COLUMNS
# ============================================

drop_cols = ['number', 'surface']

for col in drop_cols:
    if col in df.columns:
        df = df.drop(columns=[col])

# ============================================
# RENAME COLUMNS
# ============================================

df.rename(columns={
    'valid_time': 'date',
    't2m_c': 'max_temp'
}, inplace=True)

# ============================================
# DATE FEATURES
# ============================================

df['date'] = pd.to_datetime(df['date'])

df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

# Month-Day for climatology
df['month_day'] = df['date'].dt.strftime('%m-%d')

# ============================================
# SORT VALUES
# ============================================

df = df.sort_values(
    ['latitude', 'longitude', 'date']
)

# ============================================
# PREVIOUS DAY FEATURES
# ============================================

df['tmax_prev1'] = df.groupby(
    ['latitude', 'longitude']
)['max_temp'].shift(1)

df['tmax_prev2'] = df.groupby(
    ['latitude', 'longitude']
)['max_temp'].shift(2)

df['tmax_prev3'] = df.groupby(
    ['latitude', 'longitude']
)['max_temp'].shift(3)

# ============================================
# ROLLING FEATURES
# ============================================

df['rolling_3day_avg'] = (
    df.groupby(['latitude', 'longitude'])['max_temp']
    .rolling(3)
    .mean()
    .reset_index(level=[0,1], drop=True)
)

df['rolling_3day_max'] = (
    df.groupby(['latitude', 'longitude'])['max_temp']
    .rolling(3)
    .max()
    .reset_index(level=[0,1], drop=True)
)

df['rolling_3day_std'] = (
    df.groupby(['latitude', 'longitude'])['max_temp']
    .rolling(3)
    .std()
    .reset_index(level=[0,1], drop=True)
)

# ============================================
# TEMPERATURE CHANGE
# ============================================

df['temp_change_3day'] = (
    df['max_temp'] - df['tmax_prev3']
)

# ============================================
# REAL NORMAL TEMPERATURE (CLIMATOLOGY)
# ============================================

climatology = (
    df.groupby(
        ['latitude', 'longitude', 'month_day']
    )['max_temp']
    .mean()
    .reset_index()
)

climatology.rename(columns={
    'max_temp': 'normal_temp'
}, inplace=True)

# Merge climatology back
df = df.merge(
    climatology,
    on=['latitude', 'longitude', 'month_day'],
    how='left'
)

# ============================================
# TEMPERATURE DEPARTURE / ANOMALY
# ============================================

df['departure'] = (
    df['max_temp'] - df['normal_temp']
)


# ============================================
# REMOVE NaN ROWS
# ============================================

df = df.dropna()

# ============================================
# SAVE CSV
# ============================================

df.to_csv(
    "rajasthan_heatwave.csv",
    index=False
)

# ============================================
# QUICK CHECKS
# ============================================

print(df.head())

print("\nDataset Shape:")
print(df.shape)

print("\nCSV saved successfully")

        date  latitude  longitude   max_temp  year  month  day month_day  \
3 2015-04-04      23.0       69.0  27.465973  2015      4    4     04-04   
4 2015-04-05      23.0       69.0  26.699615  2015      4    5     04-05   
5 2015-04-06      23.0       69.0  27.131500  2015      4    6     04-06   
6 2015-04-07      23.0       69.0  27.939606  2015      4    7     04-07   
7 2015-04-08      23.0       69.0  30.655426  2015      4    8     04-08   

   tmax_prev1  tmax_prev2  tmax_prev3  rolling_3day_avg  rolling_3day_max  \
3   28.023346   28.123444   28.418854         27.870921         28.123444   
4   27.465973   28.023346   28.123444         27.396311         28.023346   
5   26.699615   27.465973   28.023346         27.099030         27.465973   
6   27.131500   26.699615   27.465973         27.256907         27.939606   
7   27.939606   27.131500   26.699615         28.575511         30.655426   

   rolling_3day_std  temp_change_3day  normal_temp  departure  
3          0.354